# Simple LLM evaluation with LangChain and Langfuse

In this lesson we will:

1. create and version a prompt in Langfuse,
2. use the managed prompt with LangChain and OpenRouter,
3. trace the application and read the trace,
4. evaluate the application on a small dataset.

## 1. Setup

Install the dependencies into the active notebook kernel. If the packages are installed for the first time, restart the kernel after this cell.

In [1]:
%pip install -q "langchain>=1.0,<2" "langchain-openrouter>=0.1,<1" "langfuse>=4,<5" "pandas>=2.2,<4" "python-dotenv>=1.0,<2"

/home/destiny/Projects/pydata-amsterdam/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


Create `.env` from `.env.example` before continuing.

You need a free [Langfuse Cloud](https://cloud.langfuse.com) project. Copy the public key and the secret key from **Project Settings -> API Keys** into `.env`.

In [2]:
import json
import os
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openrouter import ChatOpenRouter
from langfuse import Evaluation, get_client, observe
from langfuse.langchain import CallbackHandler

load_dotenv()

required_variables = [
    "OPENROUTER_API_KEY",
    "LANGFUSE_PUBLIC_KEY",
    "LANGFUSE_SECRET_KEY",
]
missing_variables = [name for name in required_variables if not os.getenv(name)]
if missing_variables:
    raise RuntimeError(f"Missing environment variables: {', '.join(missing_variables)}")

langfuse = get_client()
print("Langfuse connected:", langfuse.auth_check())

prompt_name = os.getenv("LANGFUSE_PROMPT", "pydata-conference-assistant")

Langfuse connected: True


## 2. Manage prompts

A prompt is an important part of an LLM application. It tells the model what to do. If a prompt is only in the code, we often need a new deployment to change it. This can be slow.

Langfuse Prompt Management keeps the prompt outside the application code. We can change a prompt and use the new version without deploying the application again. We can also:

- keep old prompt versions,
- compare two versions,
- return to an old version when there is a problem,
- use one prompt in development and another prompt in production.

Langfuse uses **labels** for this. The `production` label can point to a stable prompt for users. The `development` label can point to a new prompt that we are testing. After testing, we move the `production` label to the new version. The application code does not need to change.

First, we create a simple prompt. This creates the prompt and its first version.

In [3]:
prompt_v1 = langfuse.create_prompt(
    name=prompt_name,
    type="chat",
    prompt=[
        {"role": "system", "content": "You are a helpful conference assistant."},
        {"role": "user", "content": "{{question}}"},
    ],
    labels=["production"],
    commit_message="First version.",
)

print(f"version: {prompt_v1.version}, labels: {prompt_v1.labels}")

version: 1, labels: ['production', 'latest']


Note the `{{question}}` syntax. Langfuse uses two curly braces for input variables. LangChain uses one. We convert between them later with `get_langchain_prompt()`.

Open the **Prompts** page in Langfuse. You can see the messages, the `question` variable, and the `production` label.

### Create a second prompt version

The second version gives clearer rules. Creating it with the same name adds a new version. It does not delete version 1.

We give this version the `development` label, so the `production` label still points to version 1.

In [4]:
prompt_v2 = langfuse.create_prompt(
    name=prompt_name,
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": (
                "You are a concise conference assistant. "
                "Answer accurately in at most two sentences. "
                "If you are unsure, say that you do not know."
            ),
        },
        {"role": "user", "content": "Question: {{question}}"},
    ],
    labels=["development"],
    commit_message="Ask for short and honest answers.",
)

print(f"version: {prompt_v2.version}, labels: {prompt_v2.labels}")

version: 2, labels: ['development', 'latest']


In Langfuse, open the prompt and compare the two versions. When version 2 is ready, move the `production` label to it. No new application deployment is needed.

### Pull a managed prompt

`get_prompt` returns the `production` version by default. We pass a label to choose another one. We can also pass `version=1` to pin an exact version.

`get_langchain_prompt()` changes the Langfuse `{{question}}` syntax into the LangChain `{question}` syntax.

In [5]:
prompt_label = os.getenv("LANGFUSE_PROMPT_LABEL", "development")
managed_prompt = langfuse.get_prompt(prompt_name, label=prompt_label, type="chat")

chat_prompt = ChatPromptTemplate.from_messages(managed_prompt.get_langchain_prompt())
chat_prompt

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a concise conference assistant. Answer accurately in at most two sentences. If you are unsure, say that you do not know.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Question: {question}'), additional_kwargs={})])

## 3. Build the LangChain application

OpenRouter gives us one API for models from multiple providers. The model slug can be changed in `.env`.

Langfuse reads the token usage that OpenRouter reports. To show a cost, Langfuse matches the model name against its price table. An OpenRouter slug such as `openai/gpt-4.1-mini` may not match. If the cost is missing in the UI, add a model definition under **Settings -> Models**.

In [6]:
primary_model_name = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")
model = ChatOpenRouter(
    model=primary_model_name,
    app_title="PyData Amsterdam LLM Evaluation Tutorial",
)

chain = chat_prompt | model | StrOutputParser()

## 4. Trace the application

Langfuse has two common tracing methods:

- **Callback handler:** `CallbackHandler()` traces LangChain components. Our prompt, model, and output parser appear as separate steps. We pass it in the `config` of an invoke call.
- **Decorator:** `@observe` adds a trace to a normal Python function. This is useful for custom application logic, tools, retrieval, and other functions.

Langfuse tracing is not switched on by an environment variable. We always pass the handler, or use the decorator.

Below we use both. `@observe` gives the full application a clear name and input/output boundary. The handler traces the steps inside the chain.

In [7]:
langfuse_handler = CallbackHandler()


@observe(name="Simple QA Pipeline")
def answer_question(question: str) -> str:
    return chain.invoke(
        {"question": question},
        config={
            "callbacks": [langfuse_handler],
            "run_name": "LangChain QA Chain",
            "tags": ["tutorial", "simple", "langchain", "openrouter"],
            "metadata": {"conference": "PyData Amsterdam"},
        },
    )


question = "What is LLM evaluation, in simple terms?"
answer = answer_question(question)

langfuse.flush()

print(f"Question: {question}\n")
print(f"Answer: {answer}")

Question: What is LLM evaluation, in simple terms?

Answer: LLM evaluation is the process of testing large language models to see how well they understand and generate text. It measures their accuracy, relevance, and usefulness in different tasks.


`flush()` sends the trace immediately. In a notebook this is useful, because the kernel stays open and we want to see the trace in the UI now.

Open the trace in Langfuse. You can see the prompt, the model call, the token usage, and the latency of every step.

## 5. Evaluation basics

LLM evaluation helps us check if an application works well. We need three main parts:

- **Dataset:** A collection of test examples. Each example has an input and can have an expected output. Here, the input is a question and the expected output is the answer we expect.
- **Task:** The application that we want to test. It receives one dataset item and returns an output. Langfuse calls this the *task*.
- **Evaluator:** A function that checks the task output. It returns an `Evaluation` with a name and a value. An evaluator can use simple Python rules or another LLM.

We start with a small dataset and a simple Python evaluator. This is cheap, fast, and easy to understand.

### Load the dataset

In [8]:
dataset_candidates = [
    Path("dataset.json"),
    Path("01_simple/dataset.json"),
    Path("examples/01_simple/dataset.json"),
]
dataset_path = next((path for path in dataset_candidates if path.exists()), None)
if dataset_path is None:
    raise FileNotFoundError(
        "dataset.json not found. Start Jupyter from the repository root or the 01_simple folder."
    )

examples = json.loads(dataset_path.read_text())
examples

[{'input': {'question': 'Which city hosts PyData Amsterdam?'},
  'expected_output': {'answer': 'Amsterdam'}},
 {'input': {'question': 'What does RAG stand for?'},
  'expected_output': {'answer': 'retrieval-augmented generation'}},
 {'input': {'question': 'What numeric result does an LLM evaluator produce?'},
  'expected_output': {'answer': 'score'}}]

The file already uses the Langfuse item shape: `input`, `expected_output`, and an optional `metadata`.

### Define the task and the evaluators

A task receives `item` as a keyword argument and returns the output. We build one task per chain, so we can compare different chains later.

The first evaluator checks if the expected text is inside the model answer. It is deterministic: the same answer always gets the same result. It is useful for a first example, but it cannot understand all correct answers.

The second evaluator runs once for the whole experiment. It reports how many examples passed.

In [9]:
def create_task(chain_to_evaluate):
    def task(*, item, **kwargs) -> str:
        return chain_to_evaluate.invoke(
            {"question": item.input["question"]},
            config={"callbacks": [langfuse_handler]},
        )

    return task


def contains_reference(*, output, expected_output, **kwargs) -> Evaluation:
    expected = expected_output["answer"].casefold()
    return Evaluation(
        name="contains_reference",
        value=expected in output.casefold(),
        comment=f"Looked for {expected!r}.",
    )


def pass_rate(*, item_results, **kwargs) -> Evaluation:
    scores = [
        evaluation.value
        for item_result in item_results
        for evaluation in item_result.evaluations
        if evaluation.name == "contains_reference"
    ]
    return Evaluation(name="pass_rate", value=sum(scores) / len(scores))

## 6. Experiments

An experiment runs one application version on every dataset item. It then runs the evaluators on every output.

We will create two comparisons:

1. the same dataset and model with two different prompts,
2. the same dataset and prompt with two different models.

Each experiment changes only one thing. This is important. If we change the prompt and the model together, we cannot say which change caused the difference.

First we upload the dataset to Langfuse.

In [10]:
timestamp = datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
dataset_name = f"pydata-simple-langchain-{timestamp}"

langfuse.create_dataset(
    name=dataset_name,
    description="Introductory dataset shared by simple LLM applications.",
)
for example in examples:
    langfuse.create_dataset_item(dataset_name=dataset_name, **example)

dataset = langfuse.get_dataset(dataset_name)
print(f"{dataset.name} has {len(dataset.items)} items")

pydata-simple-langchain-20260823-130353 has 3 items


A new dataset is created on every run, so repeated runs during the workshop do not mix results.

The helper below turns an experiment result into a table, so we can read the outputs inside this notebook.

In [11]:
def results_to_frame(result) -> pd.DataFrame:
    rows = []
    for item_result in result.item_results:
        row = {
            "question": item_result.item.input["question"],
            "answer": item_result.output,
            "expected": item_result.item.expected_output["answer"],
        }
        for evaluation in item_result.evaluations:
            row[evaluation.name] = evaluation.value
        rows.append(row)
    return pd.DataFrame(rows)

### Example 1: same model, different prompts

We keep the dataset, model, and evaluators the same. We only change the prompt. This helps us measure if prompt version 2 is better than prompt version 1.

Both experiments use the prompt versions that we created in Langfuse earlier.

In [12]:
prompt_v1_template = ChatPromptTemplate.from_messages(prompt_v1.get_langchain_prompt())
prompt_v2_template = ChatPromptTemplate.from_messages(prompt_v2.get_langchain_prompt())

prompt_v1_chain = prompt_v1_template | model | StrOutputParser()
prompt_v2_chain = prompt_v2_template | model | StrOutputParser()

prompt_v1_result = dataset.run_experiment(
    name="prompt-v1",
    description="First prompt version.",
    task=create_task(prompt_v1_chain),
    evaluators=[contains_reference],
    run_evaluators=[pass_rate],
    metadata={"model": primary_model_name, "prompt": "v1"},
    max_concurrency=2,
)

prompt_v2_result = dataset.run_experiment(
    name="prompt-v2",
    description="Second prompt version.",
    task=create_task(prompt_v2_chain),
    evaluators=[contains_reference],
    run_evaluators=[pass_rate],
    metadata={"model": primary_model_name, "prompt": "v2"},
    max_concurrency=2,
)

langfuse.flush()

print(prompt_v1_result.dataset_run_url)
print(prompt_v2_result.dataset_run_url)

https://cloud.langfuse.com/project/cmt5t9q7c0k14ad0d8mobt8p7/datasets/cmt5tj7sv0smoad0dtmoli95x/runs/d512f267-22a0-4d0a-9457-da3aafe5b82d
https://cloud.langfuse.com/project/cmt5t9q7c0k14ad0d8mobt8p7/datasets/cmt5tj7sv0smoad0dtmoli95x/runs/81d8b93e-c217-41f2-ac37-69d277d6401a


In [13]:
prompt_comparison = pd.concat(
    {
        "prompt-v1": results_to_frame(prompt_v1_result),
        "prompt-v2": results_to_frame(prompt_v2_result),
    },
    names=["experiment"],
)
prompt_comparison

question  \
experiment                                                        
prompt-v1  0  What numeric result does an LLM evaluator prod...   
           1                           What does RAG stand for?   
           2                 Which city hosts PyData Amsterdam?   
prompt-v2  0  What numeric result does an LLM evaluator prod...   
           1                           What does RAG stand for?   
           2                 Which city hosts PyData Amsterdam?   

                                                         answer  \
experiment                                                        
prompt-v1  0  An LLM evaluator typically produces a numeric ...   
           1  RAG can stand for different things depending o...   
           2           PyData Amsterdam is hosted in Amsterdam.   
prompt-v2  0  An LLM evaluator produces a numeric score or r...   
           1  RAG stands for Retrieval-Augmented Generation....   
           2           PyData Amsterdam is hosted in Amsterdam.   

                                    expected  contains_reference  
experiment                                                        
prompt-v1  0                           score                True  
           1  retrieval-augmented generation                True  
           2                       Amsterdam                True  
prompt-v2  0                           score                True  
           1  retrieval-augmented generation                True  
           2                       Amsterdam                True

`format()` prints the same result as text, including the run level `pass_rate`.

In [14]:
print(prompt_v1_result.format())
print(prompt_v2_result.format())

Individual Results: Hidden (3 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: prompt-v1
📋 Run name: prompt-v1 - 2026-08-23T13:04:43.674736Z - First prompt version.
3 items
Evaluations:
  • contains_reference

Average Scores:
  • contains_reference: 1.000

Run Evaluations:
  • pass_rate: 1.000

🔗 Dataset Run:
   https://cloud.langfuse.com/project/cmt5t9q7c0k14ad0d8mobt8p7/datasets/cmt5tj7sv0smoad0dtmoli95x/runs/d512f267-22a0-4d0a-9457-da3aafe5b82d
Individual Results: Hidden (3 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: prompt-v2
📋 Run name: prompt-v2 - 2026-08-23T13:04:51.812785Z - Second prompt version.
3 items
Evaluations:
  • contains_reference

Average Scores:
  • contains_reference: 1.000

Run Evaluations:
  • pass_rate: 1.000

🔗 Dataset Run:
   https://cloud.langfuse.com/project/cmt5t9q7c0k14ad0d8mobt8p7/datasets/cmt5tj7sv0smoad0dtmoli95x

### Example 2: same prompt, different models

Now we keep the dataset, prompt version 2, and evaluators the same. We only change the model.

We already have the prompt version 2 result for the first model. We reuse it as the baseline, so we only need to run one new experiment.

In [15]:
comparison_model_name = os.getenv(
    "OPENROUTER_COMPARISON_MODEL",
    "google/gemini-2.5-flash",
)
comparison_model = ChatOpenRouter(
    model=comparison_model_name,
    app_title="PyData Amsterdam LLM Evaluation Tutorial",
)
comparison_chain = prompt_v2_template | comparison_model | StrOutputParser()

comparison_result = dataset.run_experiment(
    name=f"model-{comparison_model_name.replace('/', '-')}",
    description="Prompt version 2 on a second model.",
    task=create_task(comparison_chain),
    evaluators=[contains_reference],
    run_evaluators=[pass_rate],
    metadata={"model": comparison_model_name, "prompt": "v2"},
    max_concurrency=2,
)

langfuse.flush()

model_comparison = pd.concat(
    {
        primary_model_name: results_to_frame(prompt_v2_result),
        comparison_model_name: results_to_frame(comparison_result),
    },
    names=["experiment"],
)
model_comparison

question  \
experiment                                                                     
openai/gpt-4.1-mini     0  What numeric result does an LLM evaluator prod...   
                        1                           What does RAG stand for?   
                        2                 Which city hosts PyData Amsterdam?   
google/gemini-2.5-flash 0  What numeric result does an LLM evaluator prod...   
                        1                           What does RAG stand for?   
                        2                 Which city hosts PyData Amsterdam?   

                                                                      answer  \
experiment                                                                     
openai/gpt-4.1-mini     0  An LLM evaluator produces a numeric score or r...   
                        1  RAG stands for Retrieval-Augmented Generation....   
                        2           PyData Amsterdam is hosted in Amsterdam.   
google/gemini-2.5-flash 0  An LLM evaluator produces a numerical score or...   
                        1  RAG stands for Retrieval Augmented Generation....   
                        2           PyData Amsterdam is hosted in Amsterdam.   

                                                 expected  contains_reference  
experiment                                                                     
openai/gpt-4.1-mini     0                           score                True  
                        1  retrieval-augmented generation                True  
                        2                       Amsterdam                True  
google/gemini-2.5-flash 0                           score                True  
                        1  retrieval-augmented generation               False  
                        2                       Amsterdam                True

Open the dataset in Langfuse and compare the runs side by side.

The substring evaluator is strict. A correct answer that uses different words gets a failing score. In the next lesson we look at evaluators that can judge meaning instead of exact text.